# 🪢 CSV Data Pool Combiner

This notebook combines individual BO experiment logs into shared data-pool CSV files.

It is intentionally limited to **data pooling and syncing**. Analysis should be done in a separate notebook.

Each teammate can keep their own CSV files, such as:

- `brain_bo_usa_top3000_alice.csv`
- `brain_bo_usa_top1000_bob.csv`

This notebook combines these raw logs into date-stamped shared data pools, for example:

- `combined_brain_bo_usa_top3000_2026-04-26.csv`
- `combined_brain_bo_all_2026-04-26.csv`

In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import date
from pathlib import Path

from csv_combiner import combine_logs, find_log_files

RUN_DATE = date.today().isoformat()
RUN_DATE

# Change this if your raw CSV logs are stored somewhere else.
# Example: Path("data_alice") for alpha/data_alice
DATA_DIR = Path(".")
DATA_DIR

## Data-pool communication convention

To avoid confusion, we distinguish between two types of CSV files:

- **Raw personal logs**: generated by each teammate when running `run_one_trial(...)`.
  - Example: `brain_bo_usa_top3000_alice.csv`
  - Example: `brain_bo_usa_top3000_bob.csv`

- **Synced master data pools**: generated by this notebook after combining raw logs.
  - Example: `combined_brain_bo_usa_top3000_2026-04-26.csv`
  - Example: `combined_brain_bo_all_2026-04-26.csv`

Recommended communication rule:

> When sharing a synced data pool, always send the exact filename, date, universe, and what raw logs were included.

Example message:

> Synced master data pool updated: `combined_brain_bo_usa_top3000_2026-04-26.csv`.
> Includes Angze + Alice raw logs for USA / TOP3000 up to 2026-04-26.
> Use this file for analysis; continue appending new experiments to your own raw log.

This keeps analysis reproducible: everyone knows exactly which data snapshot was used.


## 1. Check available log files

This cell lists all raw BO log files in `DATA_DIR`.

Combined output files are ignored by the helper function.


In [ ]:
log_files = find_log_files(directory=DATA_DIR)

for file in log_files:
    print(file)

print(f"\nFound {len(log_files)} raw BO log file(s).")

## 2. Combine logs for one universe

Choose the universe you want to combine.

Common choices:
- `top3000`
- `top1000`
- `top500`
- `top200`

In [ ]:
REGION = "usa"
UNIVERSE = "top3000"

combined_path = DATA_DIR / f"combined_brain_bo_{REGION}_{UNIVERSE}_{RUN_DATE}.csv"

In [ ]:
df_universe = combine_logs(
    directory=DATA_DIR,
    region=REGION,
    universe=UNIVERSE,
    output_path=combined_path,
)

print(f"Combined shape: {df_universe.shape}")
print(f"Saved to: {combined_path}")
print("\nSuggested team message:")
print(
    f"Synced master data pool updated: `{combined_path}`. "
    f"Includes raw BO logs for {REGION.upper()} / {UNIVERSE.upper()} up to {RUN_DATE}."
)

df_universe.head()

## 3. Combine all raw logs

This combines every raw BO log file in `DATA_DIR`, regardless of universe or user.

Use this to create a full team-level data pool snapshot.

In [ ]:
combined_all_path = DATA_DIR / f"combined_brain_bo_all_{RUN_DATE}.csv"

combined_all_path

In [ ]:
df_all = combine_logs(
    directory=DATA_DIR,
    output_path=combined_all_path,
)

print(f"Combined shape: {df_all.shape}")
print(f"Saved to: {combined_all_path}")
print("\nSuggested team message:")
print(
    f"Synced master data pool updated: `{combined_all_path}`. "
    f"Includes all raw BO logs found in the current directory up to {RUN_DATE}."
)

df_all.head()

## 4. Optional: combine all universes separately

Run this cell if you want one date-stamped combined CSV for each universe.

In [ ]:
UNIVERSES = ["top3000", "top1000", "top500", "top200"]

combined_by_universe = {}

for universe in UNIVERSES:
    output_path = DATA_DIR / f"combined_brain_bo_{REGION}_{universe}_{RUN_DATE}.csv"

    try:
        df = combine_logs(
            directory=DATA_DIR,
            region=REGION,
            universe=universe,
            output_path=output_path,
        )
        combined_by_universe[universe] = df

        print(f"{universe}: saved {df.shape} to {output_path}")

    except FileNotFoundError:
        print(f"{universe}: no raw logs found; skipped")

## 5. Reminder

Generated combined CSV files are data-pool snapshots.

If you use a separate data folder such as `data_alice/`, make sure that folder is also ignored by Git if it contains private logs.

For performance summaries, rankings, and plots, use a separate analysis notebook.

Do **not** commit them to GitHub unless the team intentionally wants to share those results, because they may contain private alpha expressions and performance metrics.